# Evaluación — Módulo 4 · Tema 2: GLM con Python

**Alumno:** Erick
**Variable asignada:** `antiguedad_vehiculo_cat`  (antigüedad del vehículo)
**Fecha de entrega:** ________________

> **Instrucciones.** Este archivo evalúa las tres sesiones del tema. Las tablas ya vienen
> calculadas; tu trabajo es **responder las preguntas de interpretación** en el espacio
> "**Tu respuesta:**". Se evalúa la interpretación, no el código. Máx. 4–6 líneas por respuesta.
> Todas tus preguntas usan **tu variable asignada** (`antiguedad_vehiculo_cat`). Guarda y sube este archivo a tu repositorio.

---

## Parte 1 · Sesión 1 — Modelo de Frecuencia

**Diagnóstico del supuesto de Poisson (modelo completo):**

| métrica | valor |
| --- | --- |
| φ de Pearson | 1.1664 |
| Cameron-Trivedi α | 0.0744 |
| z | 15.80 |
| p-value | 3.7e-56 |

**Rating factors de frecuencia para `antiguedad_vehiculo_cat`** (base × RF reproduce la tasa empírica; diferencia máx = 5.2e-05):

| nivel | RF_frec | IC_inf | IC_sup | p | tasa_emp |
| --- | --- | --- | --- | --- | --- |
| (-1, 1] (ref) | 1 | 1 | 1 | 0 | 0.1698 |
| (1, 2] | 0.76 | 0.7037 | 0.8207 | 0 | 0.129 |
| (2, 3] | 0.7254 | 0.6703 | 0.785 | 0 | 0.1231 |
| (3, 4] | 0.7923 | 0.7336 | 0.8557 | 0 | 0.1345 |
| (4, 5] | 0.7934 | 0.7344 | 0.8573 | 0 | 0.1347 |
| (5, 10] | 0.8216 | 0.7718 | 0.8746 | 0 | 0.1395 |
| (10, 15] | 0.8825 | 0.826 | 0.9428 | 0.0002 | 0.1498 |
| (15, 50] | 0.6795 | 0.6113 | 0.7553 | 0 | 0.1154 |

**P1.** ¿Se cumple la equidispersión? Justifica con φ **y** con Cameron-Trivedi, y di qué familia usarías.
**Tu respuesta:** No se cumple del todo: φ = 1.166 > 1 indica que la varianza excede a la media, y Cameron-Trivedi lo confirma (α = 0.074, p ≈ 10⁻⁵⁶ ≪ 0.05), así que rechazamos la equidispersión. La sobredispersión es leve (φ < 1.5), por lo que QuasiPoisson basta —mantiene los coeficientes y corrige los errores por √φ—; no hace falta Binomial Negativa (esa se justifica con φ > 2).

**P2.** Interpreta los rating factors de tu variable: nivel más alto y más bajo, traducidos a % de
recargo/descuento. ¿Algún IC cruza 1 o tiene p > 0.05? ¿Qué harías con ese nivel?
**Tu respuesta:** La referencia es (-1,1] (RF = 1), el grupo de mayor riesgo. El RF más bajo es (15,50] con 0.679 → un descuento del 32.1% en frecuencia respecto a los menos antiguos; el más alto, después de la referencia, es (10,15] con 0.882 → −11.8%. La tendencia no es clara, ya que no necesariamente a mayor antiguedadad, se tiene una menor frecuencia. Ningún IC cruza el 1 y todos los p ≈ 0, así que mantengo los ocho niveles — cada uno diferencia riesgo de forma significativa.

**P3.** ¿Por qué el GLM one-way reproduce exactamente la tasa empírica, y qué aporta el GLM que una
tabla empírica no puede dar?
**Tu respuesta:** Porque las ecuaciones de score del Poisson con liga log y offset obligan a que, por nivel, la suma de siniestros predichos iguale la observada: eso hace exp(η) = Σn/Σe, exactamente la tasa empírica ponderada. Lo que el GLM aporta de más es combinar muchas variables de forma multiplicativa a la vez —algo que una tabla cruzada no puede— y dar un intervalo de confianza para cada rating factor.

---

## Parte 2 · Sesión 2 — Severidad y Selección de Modelos

**Comparación de modelos de frecuencia:**

| modelo | AIC | BIC | pseudoR2_McF |
| --- | --- | --- | --- |
| Poisson | 125,081.7 | 125,261.7 | 0.0198 |
| Binomial Negativa | 124,925.7 | 125,105.8 | 0.021 |

**Rating factors de severidad (Gamma) para `antiguedad_vehiculo_cat`:**

| nivel | RF_sev | severidad_emp |
| --- | --- | --- |
| (-1, 1] (ref) | 1 | 1,766 |
| (1, 2] | 0.7141 | 1,261 |
| (2, 3] | 0.6865 | 1,212 |
| (3, 4] | 0.6275 | 1,108 |
| (4, 5] | 0.7341 | 1,296 |
| (5, 10] | 0.7342 | 1,297 |
| (10, 15] | 0.74 | 1,307 |
| (15, 50] | 0.9099 | 1,607 |

**P4.** ¿Por qué se usa **Gamma** para severidad y no una regresión lineal sobre log(Y)? (menciona la
propiedad del CV y por qué Lognormal no es GLM).
**Tu respuesta:** Porque la Gamma tiene CV constante (la variabilidad relativa del monto es parecida en todos los niveles), lo que ajusta bien a la severidad de seguros. Y la Lognormal no es un GLM: no pertenece a la familia exponencial natural (su estadístico suficiente es log y, no y). Además, una regresión sobre log(Y) modela E[log Y], no E[Y], y volver a la escala original exige un factor de corrección exp(σ̂²/2). La Gamma con liga log modela E[Y] directamente, sin corrección.

**P5.** Según la tabla de comparación, ¿qué modelo elegirías? Justifica con AIC/BIC. ¿Por qué el pseudo R²
es tan bajo y eso NO significa que el modelo sea malo?
**Tu respuesta:** Elegiría Binomial Negativa: tiene menor AIC (124,926 vs 125,082) y menor BIC (125,106 vs 125,262), lo que refleja la sobredispersión leve que detectamos. El pseudo R² es bajo (~0.02) porque la ocurrencia de un siniestro tiene una enorme componente aleatoria irreducible que ningún modelo explica — no es comparable con un R² de OLS. Lo que importa es la comparación relativa entre modelos y la discriminación (Gini), no un número alto.

**P6.** Compara tus rating factors de frecuencia (Parte 1) con los de severidad para `antiguedad_vehiculo_cat`. ¿Apuntan en
la misma dirección? ¿Qué implica eso para separar Frecuencia × Severidad?
**Tu respuesta:** Apuntan en la misma dirección pero con fuerza muy distinta. En frecuencia el efecto de la antiguedad es fuerte (de RF 1.0 a 0.679, −32.1%); en severidad es mucho más plano (de 1.0 a ~0.901, –0.09). Es decir, la antiguedad predice bien cuántos siniestros, pero apenas de qué tamaño. Justo por eso se modela Frecuencia × Severidad por separado: cada componente tiene su propia estructura de riesgo y mezclarlos escondería que la antiguedad casi no mueve la severidad.

---

## Parte 3 · Sesión 3 — Validación y Tarifa

**Validación out-of-sample del modelo de frecuencia:**

| metrica | valor | ideal |
| --- | --- | --- |
| Gini (test) | 0.2315 | > 0.30 aceptable |
| Ratio pred/obs (test) | 1.0249 | ≈ 1.00 |

**Prima pura por nivel de `antiguedad_vehiculo_cat`** (Frecuencia × Severidad, con su factor de tarifa):

| nivel | prima_pura_modelo | factor_tarifa |
| --- | --- | --- |
| (-1, 1] (ref) | 305.2 | 1.6738 |
| (1, 2] | 163.76 | 0.8981 |
| (2, 3] | 148.41 | 0.814 |
| (3, 4] | 150.55 | 0.8256 |
| (4, 5] | 176.12 | 0.9659 |
| (5, 10] | 180.37 | 0.9892 |
| (10, 15] | 194.41 | 1.0662 |
| (15, 50] | 186.66 | 1.0237 |

**P7.** Interpreta las métricas de validación: ¿el modelo está bien calibrado (ratio pred/obs)? ¿discrimina
bien el riesgo (Gini)? ¿Qué mide cada una?
**Tu respuesta:** El ratio pred/obs en test es 1.025 (≈ 1), así que el modelo está bien calibrado: en promedio predice casi el total de siniestros observados. El Gini es 0.23, que mide discriminación (qué tan bien ordena de menor a mayor riesgo); está por debajo de 0.30, así que el modelo discrimina de forma modesta. Son cosas distintas: se puede estar bien calibrado (acierta el total) y discriminar poco (no separa tan bien buenos de malos riesgos).

**P8.** Lee la tabla de tarifa: ¿qué nivel de tu variable paga la prima pura más alta y cuál la más baja?
Traduce el factor de tarifa a un recargo/descuento sobre la prima promedio.
**Tu respuesta:** La prima pura más alta es la de (-1,1] con $305.2 y factor de tarifa 1.6738 → un recargo del 67.4% sobre la prima promedio. La más baja es (2,3] con $148.41 y factor 0.814 → un descuento del 18.6%. Patrón obserbado: Se tiene forma de parabola, en donde la antiguedad más baja inicia con la mayor prima, reduciendo hasta el rango (2,3] y a partir de aquí, se muestra con ligeron incremento hasta llegar nuevamente a un maximo en el rango (10,15].

**P9. (Conclusión de nota técnica).** En 3–4 líneas, redacta cómo `antiguedad_vehiculo_cat` afecta la tarifa, integrando
frecuencia, severidad y prima pura, en estilo defendible ante la CNSF.
**Tu respuesta:** La antiguedad del vehículo es un factor de tarificación de primer orden. El efecto se concentra en la frecuencia: las antiguedades de -1–1 años presentan una frecuencia esperada 1.5 veces la de 15-50 (RF 0.679; IC 95% [0.611, 0.755] para el grupo 15-50), mientras que su efecto sobre la severidad es moderado. Combinando ambos componentes, la prima pura del grupo -1-1 es 1.674 veces la prima promedio, frente a un descuento cercano al 28% en las antiguedades de (2,3]. Se recomienda conservar la antiguedad segmentada en las ocho bandas, por su fuerte poder de diferenciación y su consistencia actuarial.

---
*Evaluación generada automáticamente · Diplomado ML en Seguros · FC UNAM · Módulo 4 · Tema 2*
